# Okuyucu Ajanı — Ham model OCR ölçümü

Bu notebook, ince ayar uygulanmamış Qwen2.5-VL-7B-Instruct modelinin OCR başarımını 100 örneklik test kümesi üzerinde ölçer. İnce ayar notebook'u **aynı test klasörünü** kullanır; böylece öncesi–sonrası karşılaştırması aynı koşullarda yapılır.

**Adımlar**
1. Kütüphaneleri kur
2. Veri kümesini aç ve `test/` klasörünü bul
3. Modeli ve işlemciyi yükle
4. Yardımcı fonksiyonları tanımla: görsel yükleme, çıkarım, CER, Türkçe karakter hata oranı
5. İlk üç örnekte çıktıyı gözle doğrula
6. 100 örneğin tamamını işle
7. Bozulma grubu (A/B/C/D) bazında özet çıkar
8. Sonuçları kaydet ve özet tabloyu yazdır

> Bozulma grupları: **A** temiz baskı · **B** hafif bulanıklık · **C** belirgin bulanıklık ve gürültü · **D** düşük çözünürlük, ağır gürültü. Her gruptan bir örnek depoda: `QWEN_VL_ocr/data/`.

**Kaynak metin:** Hugging Face `erdem-erdem/Turkish-Law-Documents-700k-clustered` — Yargıtay ve Danıştay'ın kamuya açık karar metinleri, belge sayfası olarak yeniden basılır.

**Sonuç:** Bu ölçümde CER 0,4283 çıkar; aynı test kümesinde ince ayarlı model 0,1076 değerine ulaşır.

## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Gerekli kütüphaneler:
# - transformers: Qwen2.5-VL model/processor sınıfları için (Qwen2.5-VL desteği transformers>=4.49 ile geldi)
# - qwen-vl-utils: görsel/video ön işleme (process_vision_info) için resmi yardımcı paket
# - jiwer: CER (Character Error Rate) hesabı ve karakter bazlı hizalama (alignment) için
# - accelerate: device_map="auto" ile çok GPU/otomatik yerleştirme için
!pip install -q -U transformers accelerate qwen-vl-utils jiwer

In [ ]:
import os
import json
import glob
import time
import re
import zipfile

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import jiwer

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Veri kümesini yerel diske aç

İnce ayar notebook'uyla aynı arşiv kullanılır. Yüzlerce küçük PNG'yi ağ bağlantılı diskten okumak yavaş olduğundan arşiv bir kez yerel diske açılır. Değerlendirme `test/` klasöründeki 100 örnek üzerinden yapılır.

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/qwen_vl_7b_model"
DATASET_ZIP_PATH = "/content/drive/MyDrive/dataset_1000.zip"
LOCAL_DATASET_ROOT = "/content/dataset_1000"
RESULTS_PATH = "/content/drive/MyDrive/baseline_results.json"

assert os.path.isdir(MODEL_PATH), f"Model klasörü bulunamadı: {MODEL_PATH}"
assert os.path.isfile(DATASET_ZIP_PATH), f"dataset_1000.zip bulunamadı: {DATASET_ZIP_PATH}"

if not os.path.isdir(LOCAL_DATASET_ROOT):
    os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)
    print("dataset_1000.zip çıkartılıyor (bir kereye mahsus, birkaç dakika sürebilir)...")
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_DATASET_ROOT)
    print("Bitti.")
else:
    print("Local dataset klasörü zaten mevcut, çıkartma atlandı.")


def _find_split_root(root: str) -> str:
    entries = [e for e in os.listdir(root) if not e.startswith(".")]
    has_splits = any(e.lower() in ("train", "valid", "val", "test") for e in entries)
    if has_splits:
        return root
    if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
        return _find_split_root(os.path.join(root, entries[0]))
    return root


LOCAL_DATASET_ROOT = _find_split_root(LOCAL_DATASET_ROOT)


def _resolve_split_dir(root: str, names: list) -> str:
    for name in names:
        for candidate in (name, name.lower(), name.capitalize()):
            path = os.path.join(root, candidate)
            if os.path.isdir(path):
                return path
    raise FileNotFoundError(f"{names} isimlerinden hiçbiri {root} altında bulunamadı: {os.listdir(root)}")


TEST_DIR = _resolve_split_dir(LOCAL_DATASET_ROOT, ["test"])

augmentation_log = {}
for meta_path in (
    os.path.join(LOCAL_DATASET_ROOT, "dataset_metadata.json"),
    os.path.join(TEST_DIR, "augmentation_log.json"),
    os.path.join(TEST_DIR, "dataset_metadata.json"),
):
    if os.path.isfile(meta_path):
        with open(meta_path, "r", encoding="utf-8") as f:
            augmentation_log = json.load(f)
        print(f"Metadata yüklendi: {meta_path} ({len(augmentation_log)} kayıt)")
        break
else:
    print("Metadata dosyası yok; grup bilgisi dosya adından alınacak.")

print(f"TEST_DIR = {TEST_DIR}")
print("Dataset kök içeriği:", os.listdir(LOCAL_DATASET_ROOT))

## 3. Modeli ve işlemciyi yükle

Model bfloat16 hassasiyetiyle ve `device_map="auto"` ile, ince ayar uygulanmamış hâliyle yüklenir. 7B boyutundaki bir görsel-dil modeli için yeterli GPU belleği gerekir.

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("Model ve processor yüklendi.")

## 4. OCR çıkarım fonksiyonu

Modele, belgeyi harfi harfine ve eksiksiz biçimde yazıya dökmesini isteyen Türkçe bir talimat verilir. Talimat; biçimlendirme veya yorum eklenmemesini, yalnızca belgedeki metnin birebir yazılmasını belirtir.

In [ ]:
OCR_PROMPT = (
    "Bu belgedeki metni eksiksiz ve doğru şekilde transkribe et. "
    "Sadece belgede yazan metni, satır satır ve orijinaline sadık kalarak yaz. "
    "Yorum, açıklama veya başlık ekleme; markdown biçimlendirmesi kullanma."
)


def run_ocr(image_path: str, max_new_tokens: int = 2048) -> str:
    """Tek bir görüntü için Qwen2.5-VL modelinden OCR çıktısı üretir."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": OCR_PROMPT},
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Girdi olarak verilen prompt token'larını çıktıdan çıkar, sadece üretilen kısmı bırak
    trimmed_ids = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return output_text.strip()

## 5. Metrik fonksiyonları

- **CER (karakter hata oranı):** `jiwer.cer` ile hesaplanır — ekleme, silme ve değiştirme sayısının referans karakter sayısına oranı.
- **Türkçe karakter hata oranı:** `ı, i, ş, s, ğ, g, ö, o, ü, u, ç, c` harflerinin (büyük/küçük ve İ/I dâhil) doğru üretilme oranını ölçer. `jiwer.process_characters` ile referans metin ve model çıktısı karakter düzeyinde hizalanır; her referans karakteri doğru, başka bir karakterle karışmış ya da hiç üretilmemiş olarak işaretlenir. Türkçe'ye özgü karakterlerin kaçının doğru üretilmediği sayılarak oran bulunur.

In [ ]:
TURKISH_CHARS = set("ıiİIşsğgöoüuçc")


def compute_cer(reference: str, hypothesis: str) -> float:
    if len(reference) == 0:
        return 0.0 if len(hypothesis) == 0 else 1.0
    return jiwer.cer(reference, hypothesis)


def compute_turkish_char_error_rate(reference: str, hypothesis: str) -> dict:
    """Türkçeye özgü karakterler için pozisyon bazlı hata oranı hesaplar.

    Referans karakterlerini jiwer'in karakter hizalamasına göre 'equal',
    'substitute' veya 'delete' olarak sınıflandırır; TURKISH_CHARS kümesindeki
    referans karakterlerinden kaçının doğru üretildiğini sayar.
    """
    if len(reference) == 0:
        return {"turkish_char_error_rate": 0.0, "turkish_char_total": 0, "turkish_char_errors": 0}

    output = jiwer.process_characters(reference, hypothesis)
    alignment = output.alignments[0]

    total = 0
    errors = 0
    for chunk in alignment:
        if chunk.type == "insert":
            # Hipotezde fazladan üretilmiş karakter, referansta karşılığı yok -> atla
            continue
        ref_slice = reference[chunk.ref_start_idx:chunk.ref_end_idx]
        for i, ref_char in enumerate(ref_slice):
            if ref_char not in TURKISH_CHARS:
                continue
            total += 1
            if chunk.type != "equal":
                errors += 1

    rate = (errors / total) if total > 0 else 0.0
    return {
        "turkish_char_error_rate": rate,
        "turkish_char_total": total,
        "turkish_char_errors": errors,
    }

## 6. Test örneklerini listele

`test/` klasöründeki PNG dosyaları eğitimde kullanılmaz. Beklenen sayı: 100.

In [ ]:
png_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.png")))
sample_ids = [os.path.splitext(os.path.basename(p))[0] for p in png_files]
print(f"{len(sample_ids)} örnek bulundu.")
print(sample_ids[:5], "...", sample_ids[-5:])

## 7. Pilot çalıştırma: ilk üç örnek

100 örneğin tamamına geçmeden önce çıktı gözle kontrol edilir.

In [ ]:
def load_ground_truth(sample_id: str) -> str:
    txt_path = os.path.join(TEST_DIR, f"{sample_id}.txt")
    with open(txt_path, "r", encoding="utf-8") as f:
        return f.read()


def evaluate_sample(sample_id: str) -> dict:
    image_path = os.path.join(TEST_DIR, f"{sample_id}.png")
    ground_truth = load_ground_truth(sample_id)

    start = time.time()
    prediction = run_ocr(image_path)
    elapsed = time.time() - start

    cer = compute_cer(ground_truth, prediction)
    turkish_stats = compute_turkish_char_error_rate(ground_truth, prediction)
    meta = augmentation_log.get(sample_id, {})

    return {
        "id": sample_id,
        "group": meta.get("group", sample_id.split("_")[0]),
        "cer": cer,
        "turkish_char_error_rate": turkish_stats["turkish_char_error_rate"],
        "turkish_char_total": turkish_stats["turkish_char_total"],
        "turkish_char_errors": turkish_stats["turkish_char_errors"],
        "ground_truth_len": len(ground_truth),
        "prediction_len": len(prediction),
        "inference_seconds": elapsed,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "metadata": meta,
    }


pilot_ids = sample_ids[:3]
pilot_results = []
for sid in pilot_ids:
    print(f"--- {sid} işleniyor ---")
    res = evaluate_sample(sid)
    pilot_results.append(res)
    print(f"CER: {res['cer']:.4f} | Türkçe karakter hata oranı: {res['turkish_char_error_rate']:.4f} "
          f"({res['turkish_char_errors']}/{res['turkish_char_total']}) | süre: {res['inference_seconds']:.1f}s")
    print("Ground truth (ilk 300 karakter):\n", res["ground_truth"][:300])
    print("Model çıktısı (ilk 300 karakter):\n", res["prediction"][:300])
    print()


**Kontrol noktası:** Yukarıdaki üç çıktı belge içeriğine yakınsa (boş ya da anlamsız metin değilse) bir sonraki hücreyle 100 örneğin tamamına geçebilirsiniz.

## 8. Tüm test kümesini işle

In [ ]:
all_results = []
for idx, sid in enumerate(sample_ids, start=1):
    print(f"[{idx}/{len(sample_ids)}] {sid} işleniyor...")
    res = evaluate_sample(sid)
    all_results.append(res)
    print(f"  CER: {res['cer']:.4f} | Türkçe char hata: {res['turkish_char_error_rate']:.4f} | süre: {res['inference_seconds']:.1f}s")

print("\nTüm örnekler işlendi.")

## 9. Grup bazında özet ve en kötü beş örnek

In [ ]:
def summarize_group(results: list) -> dict:
    n = len(results)
    avg_cer = sum(r["cer"] for r in results) / n
    total_tr_chars = sum(r["turkish_char_total"] for r in results)
    total_tr_errors = sum(r["turkish_char_errors"] for r in results)
    avg_tr_error_rate_per_sample = sum(r["turkish_char_error_rate"] for r in results) / n
    pooled_tr_error_rate = (total_tr_errors / total_tr_chars) if total_tr_chars > 0 else 0.0
    return {
        "num_samples": n,
        "avg_cer": avg_cer,
        "avg_turkish_char_error_rate": avg_tr_error_rate_per_sample,
        "pooled_turkish_char_error_rate": pooled_tr_error_rate,
    }


genel_ozet = summarize_group(all_results)

gruplar = sorted(set(r["group"] for r in all_results))
grup_bazli_sonuclar = {}
for g in gruplar:
    group_results = [r for r in all_results if r["group"] == g]
    grup_bazli_sonuclar[g] = summarize_group(group_results)

worst5 = sorted(all_results, key=lambda r: r["cer"], reverse=True)[:5]
en_kotu_5 = [
    {
        "id": r["id"],
        "group": r["group"],
        "cer": r["cer"],
        "turkish_char_error_rate": r["turkish_char_error_rate"],
        "ground_truth_preview": r["ground_truth"][:200],
        "prediction_preview": r["prediction"][:200],
    }
    for r in worst5
]

print("Genel özet:", genel_ozet)
print("\nGrup bazlı sonuçlar:")
for g, s in grup_bazli_sonuclar.items():
    print(f"  {g}: {s}")
print("\nEn kötü 5 örnek:")
for r in en_kotu_5:
    print(f"  {r['id']} (grup {r['group']}) - CER={r['cer']:.4f}")

## 10. Sonuçları kaydet

In [ ]:
ornek_bazli_detaylar = [
    {
        "id": r["id"],
        "group": r["group"],
        "cer": r["cer"],
        "turkish_char_error_rate": r["turkish_char_error_rate"],
        "turkish_char_total": r["turkish_char_total"],
        "turkish_char_errors": r["turkish_char_errors"],
        "ground_truth_len": r["ground_truth_len"],
        "prediction_len": r["prediction_len"],
        "inference_seconds": r["inference_seconds"],
        "ground_truth": r["ground_truth"],
        "prediction": r["prediction"],
        "metadata": r["metadata"],
    }
    for r in all_results
]

final_output = {
    "genel_ozet": genel_ozet,
    "grup_bazli_sonuclar": grup_bazli_sonuclar,
    "en_kotu_5_ornek": en_kotu_5,
    "ornek_bazli_detaylar": ornek_bazli_detaylar,
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

print(f"Sonuçlar kaydedildi: {RESULTS_PATH}")

## 11. Özet tablo

In [ ]:
header = f"{'Grup':<8}{'Örnek Sayısı':<14}{'Ort. CER':<12}{'Ort. TR Char Hata Oranı':<26}"
print(header)
print("-" * len(header))
for g in gruplar:
    s = grup_bazli_sonuclar[g]
    print(f"{g:<8}{s['num_samples']:<14}{s['avg_cer']:<12.4f}{s['avg_turkish_char_error_rate']:<26.4f}")
print("-" * len(header))
print(f"{'GENEL':<8}{genel_ozet['num_samples']:<14}{genel_ozet['avg_cer']:<12.4f}{genel_ozet['avg_turkish_char_error_rate']:<26.4f}")